# Hexaly Max Clique Benchmarking

In [1]:
import numpy as np
import os
import time
import hexaly.optimizer
from maxclique import read_dimacs_clq, build_clique_hamiltonian
from maxclique import clique_number_from_energy, extract_clique
from maxclique import greedy_clique_from_weights

### Set up Hexaly solver

In [2]:
def hexaly_solver(Q,c,sum_constraint, time_limit= 1800):
    n = len(np.squeeze(c))
    def external_xTQx(x):
        return x @ Q @ x

    def external_2Qx(x):
        return 2 * Q @ x
    
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        model = optimizer.model
        x = [model.float(0.0,100.0) for _ in range(n)]
        linear_term = model.sum(c[i]*x[i] for i in range(n))
        xTQx_func = model.create_double_external_function(external_xTQx, external_2Qx)
        objective = linear_term + xTQx_func(x)

        model.constraint(model.sum(x) == sum_constraint)
        model.minimize(objective)
        model.close()

        optimizer.param.time_limit = time_limit
        print("Starting Hexaly solver...")
        optimizer.solve()
        solution = [x[i].value for i in range(n)]
        objective_val = objective.value
        print(f"Status: {optimizer.solution.status}")
        return solution, objective_val


### load data file

In [3]:
instance_dir = "Instances/"
instance_name = "keller4.clq"
instance_path = os.path.join(instance_dir, instance_name)
try:
    A, n, m = read_dimacs_clq(instance_path)
    print("file loaded sucessfully")
except FileNotFoundError:
    print(f"{instance_path} does not exist")

file loaded sucessfully


In [4]:
name = os.path.splitext(instance_name)[0]
print(f"{name}: n = {n} vertices, m = {m} edges (declared)")
print(f"A shape: {A.shape}")
print(f"edges in A: {int(A.sum()) // 2}")
print(f"symmetric: {np.array_equal(A, A.T)}")
print(f"self-loops: {int(np.trace(A))}")
print(f"density: {A.sum() / (n * (n - 1)):.5f}")

keller4: n = 171 vertices, m = 9435 edges (declared)
A shape: (171, 171)
edges in A: 9435
symmetric: True
self-loops: 0
density: 0.64912


In [5]:
c = np.zeros(n)
H = build_clique_hamiltonian(A)

sum_constraint = 1
time_limit = 300

In [6]:
hexaly_start = time.time()
hex_sol_list, hex_obj = hexaly_solver(Q=H,
                                      c=c,
                                      sum_constraint=sum_constraint,
                                      time_limit=time_limit)
hexaly_end = time.time()
hexaly_time = hexaly_end-hexaly_start
omega = clique_number_from_energy(hex_obj, sum_constraint)
support, isclique = extract_clique(hex_sol_list, A)
greedy = greedy_clique_from_weights(hex_sol_list, A)

print(f"{'energy':>16}{'omega_est':>12}{'clique':>9}{'time (s)':>12}")
print(f"{hex_obj:>16.6f}{omega:>12.3f}{len(greedy):>9}{hexaly_time:>12.3f}")


Preprocess model 100%Starting Hexaly solver...
Push initial solution 100%
Model:  expressions = 351, decisions = 171, constraints = 1, objectives = 1
Param:  time limit = 300 sec, no iteration limit

[objective direction ]:     minimize

[  0 sec,       0 itr]: No feasible solution found (infeas = 1)
[  1 sec,       0 itr]:    -0.857131
[  2 sec,    3325 itr]:    -0.857131
[  3 sec,    3325 itr]:    -0.857131
[  4 sec,    4314 itr]:    -0.857131
[  5 sec,    5223 itr]:    -0.857131
[  6 sec,    6774 itr]:    -0.857131
[  7 sec,    7867 itr]:    -0.857131
[  8 sec,    9437 itr]:    -0.857131
[  9 sec,   10994 itr]:    -0.862687
[ 10 sec,   13238 itr]:    -0.876333
[ optimality gap     ]:      100.00%
[ 11 sec,   15235 itr]:    -0.877773
[ 12 sec,   17245 itr]:      -0.8859
[ 13 sec,   19343 itr]:     -0.90005
[ 14 sec,   21370 itr]:    -0.903539
[ 15 sec,   21370 itr]:    -0.903539
[ 16 sec,   25828 itr]:    -0.905469
[ 17 sec,   25828 itr]:    -0.905469
[ 18 sec,   30851 itr]:    -0.90